## Uma introdução prática ao pandas 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'].insert(0, 'SimHei')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
%config InlineBackend.figure_format = 'svg'

### Dinamização de dados

1. Agregação de dados (cálculos métricos)
2. Classificação e valores principais
3. Tabelas dinâmicas e tabelas cruzadas

In [ ]:
sales_df = pd.read_excel('res/2020-sales-data.xlsx', sheet_name='data')
sales_df.head(5)

In [ ]:
sales_df.info()

In [ ]:
# Adicione colunas de valor de vendas, lucro bruto e mês
sales_df['sales_amount'] = sales_df.unit_price * sales_df.quantity_sold
sales_df['gross_profit'] = sales_df.sales_amount - sales_df.direct_cost
sales_df['month'] = sales_df.sale_date.dt.month
sales_df.head(5)

In [ ]:
def make_tag(price):
    if price < 300:
        return 'entry-level'
    elif price < 800:
        return 'mid-range'
    return 'high-end'

In [ ]:
# Adicione um rótulo de faixa de preço com base no preço do produto
sales_df['price_band'] = sales_df.unit_price.apply(make_tag)
sales_df.head(5)

In [ ]:
# Calcule as métricas da estrela norte
GMV, profit, quantity = sales_df[['sales_amount', 'gross_profit', 'quantity_sold']].sum()
print(f'sales amount: {GMV} yuan')
print(f'gross profit: {profit} yuan')
print(f'quantity sold: {quantity} items')
print(f'gross margin: {profit / GMV:.2%}')

In [ ]:
# Calcule o valor das vendas e o lucro bruto de cada mês
temp1 = sales_df.groupby('month')[['sales_amount', 'gross_profit']].agg('sum')
temp1

In [ ]:
# Use uma tabela dinâmica para calcular o valor das vendas e o lucro bruto de cada mês
pd.pivot_table(
    sales_df,
    index='month',
    values=['sales_amount', 'gross_profit'],
    aggfunc='sum'
)

In [ ]:
# Trace um gráfico de linhas
temp1.plot(
    kind='line',
    figsize=(10, 5),
    y=['sales_amount', 'gross_profit'],   # dados mostrados no eixo y
    xlabel='',              # rótulo do eixo x
    ylabel='sales amount and gross profit',  # rótulo para o eixo y
    marker='^',             # símbolo de marcador
)
# plt.fill_between(np.arange(1, 13), temp1.sales_amount, onde=temp1.sales_amount >= 3e6, facecolor='vermelho', alfa=0,25)
# plt.fill_between(np.arange(1, 13), temp1.sales_amount, onde=temp1.sales_amount < 3e6, facecolor='verde', alfa=0,25)
# Personalize o intervalo do eixo y
plt.ylim(0, 6e6)
# Personalize os ticks do eixo x
plt.xticks(np.arange(1, 13), labels=[f'Month {x}' for x in range(1, 13)])
# Personalize o título
plt.title('Monthly Sales Amount and Gross Profit in 2020', fontdict={'fontsize': 22, 'color': 'navy'})
plt.show()

In [ ]:
plt.cm.RdYlBu_r

In [ ]:
# Calcular a variação mês a mês
temp1['sales_amount_mom'] = temp1.sales_amount.pct_change()
temp1['gross_profit_mom'] = temp1.gross_profit.pct_change()
# Reordenar o índice
temp1 = temp1.reindex(columns=['sales_amount', 'sales_amount_mom', 'gross_profit', 'gross_profit_mom'])
# Renderizar a saída
temp1.style.format(
    formatter={
        'sales_amount_mom': '{:.2%}',
        'gross_profit_mom': '{:.2%}'
    },
    na_rep='-------'
).background_gradient(
    'RdYlBu_r',
    subset=['sales_amount_mom', 'gross_profit_mom']
)

In [ ]:
# Trace um gráfico de linhas horizontais
mu = temp1.sales_amount.mean()
temp1['diff'] = temp1.sales_amount - mu
temp1['colors'] = temp1.sales_amount.map(lambda x: 'green' if x > mu else 'red')

plt.figure(figsize=(8, 6), dpi=200)
plt.hlines(y=temp1.index, xmin=0, xmax=temp1['diff'], color=temp1.colors, alpha=0.6, linewidth=6)
plt.yticks(np.arange(1, 13), labels=[f'Month {x}' for x in np.arange(1, 13)])
# Personalize as linhas de grade
plt.grid(linestyle='--', linewidth=0.4, alpha=0.5)
plt.show()

In [ ]:
# Participação de contribuição de cada marca no valor total das vendas
temp2 = sales_df.groupby('brand')['sales_amount'].sum()
temp2.plot(
    kind='pie',
    ylabel='',
    autopct='%.2f%%',  # calcular e mostrar porcentagens automaticamente
    pctdistance=0.82,  # distância dos rótulos de porcentagem ao centro
    wedgeprops=dict(width=0.35, edgecolor='w'),  # personalizar o gráfico de pizza em estilo donut
    explode=[0.1, 0, 0, 0, 0],  # explodir o gráfico de pizza
)
plt.show()

In [ ]:
# Valor de vendas para cada região de vendas por mês
temp3 = sales_df.groupby(['sales_region', 'month'], as_index=False)[['sales_amount']].sum()
# pivot - gira linhas em colunas (tabela estreita -> tabela larga)
# derreter - girar colunas em linhas (tabela larga -> tabela estreita)
temp3.pivot(index='sales_region', columns='month', values='sales_amount').fillna(0).astype('i8')

In [ ]:
# Crie uma tabela dinâmica
pd.pivot_table(
    sales_df,
    index='sales_region',
    columns='month',
    values='sales_amount',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

In [ ]:
# Converta o campo de faixa de preço para o tipo de categoria e especifique a ordem de classificação
sales_df['price_band'] = sales_df.price_band.astype('category').cat.reorder_categories(['high-end', 'mid-range', 'entry-level'])
sales_df.info()

In [ ]:
# Calcule o volume de vendas mensal para produtos em cada faixa de preço
temp4 = sales_df.pivot_table(
    index='price_band',
    columns='month',
    values='quantity_sold',
    observed=False,
    fill_value=0,
    aggfunc='sum'
)
temp4

In [ ]:
# Tabela cruzada
pd.crosstab(
    index=sales_df.price_band,
    columns=sales_df.month,
    values=sales_df.quantity_sold,
    aggfunc='sum'
)

In [ ]:
blood_types = np.array(['B', 'A', 'O', 'O', 'AB', 'B', 'O', 'B', 'AB', 'A', 'A', 'O', 'B', 'O', 'O', 'O', 'O', 'A', 'B', 'B'])
personality_types = np.array(['𝛃', '𝛂', '𝛂', '𝛂', '𝛃', '𝛂', '𝛄', '𝛄', '𝛂', '𝛄', '𝛃', '𝛂', '𝛂', '𝛂', '𝛄', '𝛄', '𝛂', '𝛂', '𝛂', '𝛂'])

# Criar uma tabela cruzada
pd.crosstab(
    index=blood_types,
    columns=personality_types,
    rownames=['blood_type'],
    colnames=['personality'],
)

In [ ]:
# Trace um gráfico de barras empilhadas
temp4.T.plot(
    figsize=(10, 4),
    kind='bar',
    width=0.6,
    xlabel='',
    ylabel='quantity sold',
    stacked=True
)
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Divida cada valor pela quantidade total vendida no mês correspondente
temp5 = temp4.T.divide(temp4.sum(), axis=0)
temp5

In [ ]:
# Trace um gráfico de barras empilhadas com porcentagem
temp5.plot(
    figsize=(10, 4),
    kind='bar',
    width=0.6,
    xlabel='',
    ylabel='sales volume share',
    stacked=True
)
plt.xticks(rotation=0)
plt.yticks(np.linspace(0, 1, 6), labels=[f'{x:.0%}' for x in np.linspace(0, 1, 6)])
plt.legend(loc='lower center')

for i in temp5.index:
    y1, y2, y3 = temp5.loc[i]
    plt.text(i - 1, y2 / 2 + y1, f'{y2:.2%}', ha='center', va='center', fontdict={'size': 8})
    plt.text(i - 1, y3 / 2 + y2 + y1, f'{y3:.2%}', ha='center', va='center', fontdict={'size': 8})

plt.show()

### Exercício: análise de postos de trabalho

1. Conte o número de cidades, registros de empregos, cargos e o salário médio mensal.
2. Classifique o número de posições em cada cidade, de maior para menor.
3. Classifique o salário médio em cada cidade, do maior para o menor.
4. Calcule a parcela dos requisitos de educação nas ofertas de emprego.
5. Calcule a parcela dos requisitos de experiência profissional nas ofertas de emprego.
6. Analise a relação entre salário, escolaridade e experiência profissional.

In [ ]:
jobs_df = pd.read_csv('res/cleaned_jobs.csv')
jobs_df

In [ ]:
# Calcule as métricas da estrela norte
city_count = jobs_df['city'].nunique()
info_count = jobs_df['company_name'].count()
post_count = jobs_df['pos_count'].sum()
salary_avg = jobs_df['salary'].mean().round(1)
print(f'number of cities: {city_count}')
print(f'number of records: {info_count}')
print(f'number of positions: {post_count}')
print(f'average salary: {salary_avg}')

In [ ]:
# Classifique o número de posições em cada cidade de maior para menor
jobs_df.groupby('city')[['pos_count']].sum().sort_values(by='pos_count', ascending=False)

In [ ]:
pd.pivot_table(
    jobs_df,
    index='city',
    values='pos_count',
    aggfunc='sum'
).sort_values(by='pos_count', ascending=False)

In [ ]:
jobs_df.groupby('city')[['salary']].mean().round(1).sort_values(by='salary', ascending=False)

In [ ]:
# Classifique o salário médio em cada cidade do maior para o menor
pd.pivot_table(
    jobs_df,
    index='city',
    values='salary',
    aggfunc='mean'
).round(1).sort_values(by='salary', ascending=False)

In [ ]:
jobs_df['edu'] = jobs_df.edu.astype('category').cat.reorder_categories(['No Requirement', 'Associate Degree', 'Bachelor Degree', 'Graduate Degree'])

In [ ]:
# Calcular a parcela dos requisitos de educação nas ofertas de emprego
pd.pivot_table(
    jobs_df,
    index='edu',
    values='pos_count',
    aggfunc='sum',
    observed=True
).plot(
    kind='pie',
    ylabel='',
    subplots=True,
    legend=False,
    autopct='%.2f%%',
    pctdistance=0.85,
    wedgeprops={'width': 0.35}
)
plt.show()

In [ ]:
jobs_df['year'] = jobs_df.year.astype('category').cat.reorder_categories(['New Graduate', 'Within 1 Year', 'No Experience Limit', '1-3 Years', '3-5 Years', '5+ Years'])

In [ ]:
# Trace a distribuição dos requisitos de experiência profissional nas ofertas de emprego
pd.pivot_table(
    jobs_df,
    index='year',
    values='pos_count',
    aggfunc='sum',
    observed=True
).plot(
    kind='pie',
    y='pos_count',
    ylabel='',
    legend=False,
    autopct='%.2f%%',
    pctdistance=0.85,
    wedgeprops={'width': 0.35}
)
plt.show()

In [ ]:
# Calcule o salário médio por nível de escolaridade e experiência profissional
temp6 = pd.pivot_table(
    jobs_df,
    index='edu',
    columns='year',
    values='salary',
    observed=False,
    fill_value=0
).round(1)
temp6

In [ ]:
# Trace um mapa de calor
plt.imshow(temp6, cmap='Reds')
plt.xticks(np.arange(6), labels=temp6.columns)
plt.yticks(np.arange(4), labels=temp6.index)

for i in range(temp6.index.size):
    for j in range(temp6.columns.size):
        value = temp6.iat[i, j]
        color = 'w' if value > salary_avg else 'k'
        plt.text(j, i, value, ha='center', va='center', color=color)

# Personalize a barra de cores
plt.colorbar()
plt.show()

In [ ]:
# % pip instalar o Seaborn

In [ ]:
import seaborn as sns

sns.heatmap(temp6, cmap='Reds', annot=True)
plt.xlabel('')
plt.ylabel('')
plt.yticks(rotation=0)
plt.show()